In [ ]:
import cv2
import mediapipe as mp

# Initialize mediapipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# Finger tip indices (thumb, index, middle, ring, pinky)
fingertips = [4, 8, 12, 16, 20]

# Count fingers
def count_fingers(lm):
    count = 0
    # Thumb: check if tip x > previous joint x (for right hand) or < (for left)
    if lm[4].x < lm[3].x:  # this logic assumes palm is facing the camera
        count += 1
    for tip in fingertips[1:]:
        if lm[tip].y < lm[tip - 2].y:
            count += 1
    return count

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    left_fingers = 0
    right_fingers = 0

    if results.multi_hand_landmarks and results.multi_handedness:
        for i, hand_landmarks in enumerate(results.multi_hand_landmarks):
            label = results.multi_handedness[i].classification[0].label  # 'Left' or 'Right'
            lm = hand_landmarks.landmark

            # Count fingers
            fingers = count_fingers(lm)

            if label == 'Left':
                left_fingers = fingers
            else:
                right_fingers = fingers

            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    # Calculate result
    result = left_fingers - right_fingers

    # Display
    cv2.putText(frame, f"Left: {left_fingers} | Right: {right_fingers}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2)

    cv2.putText(frame, f"Result: {result}", (10, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)

    cv2.imshow("Two-Hand Subtraction", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
